In [4]:
# Step 1 — Import libraries
import pandas as pd
import sqlite3
import os

# Step 2 — Load the 3 CSV files (with encoding fix)
brent = pd.read_csv('brent_prices.csv', encoding='latin1')
iran = pd.read_csv('iran_production.csv', encoding='latin1')
events = pd.read_csv('events.csv', encoding='latin1')

# Step 3 — Preview each file
print("=== BRENT OIL PRICES ===")
print(brent.head())
print("\n=== IRAN PRODUCTION ===")
print(iran.head())
print("\n=== EVENTS ===")
print(events.head())

=== BRENT OIL PRICES ===
           date  price
0  May 20, 1987  18.63
1  May 21, 1987  18.45
2  May 22, 1987  18.55
3  May 25, 1987  18.60
4  May 26, 1987  18.63

=== IRAN PRODUCTION ===
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                   ï»¿Report generated on: 04-06-2026 16:31:52
API                  nan                                                1973.

In [11]:
# ============================================
# STEP 2 — CLEAN ALL 3 DATASETS
# ============================================

# --- Clean Brent Prices ---
brent.columns = ['date', 'price']
brent['date'] = pd.to_datetime(brent['date'], format='mixed')  # ← fixed
brent['price'] = pd.to_numeric(brent['price'], errors='coerce')
brent = brent.dropna()
brent = brent[brent['date'] >= '2017-01-01']  # focus on tension period
print("✅ Brent Prices cleaned:")
print(f"   Rows: {len(brent)}")
print(f"   Date range: {brent['date'].min()} to {brent['date'].max()}")
print(brent.head(3))

✅ Brent Prices cleaned:
   Rows: 2345
   Date range: 2017-01-03 00:00:00 to 2026-03-30 00:00:00
           date  price
7516 2017-01-03  55.05
7517 2017-01-04  54.57
7518 2017-01-05  54.99


In [9]:
# Build Iran production dataframe directly from the data we already have
import pandas as pd

iran_data = {
    'year': [2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025],
    'production_mbd': [4438.25, 4949.40, 4711.35, 3400.32, 3483.97, 3682.84, 4119.43, 4626.73, 4711.30]
}

iran_clean = pd.DataFrame(iran_data)

print("✅ Iran Production cleaned:")
print(iran_clean)

✅ Iran Production cleaned:
   year  production_mbd
0  2017         4438.25
1  2018         4949.40
2  2019         4711.35
3  2020         3400.32
4  2021         3483.97
5  2022         3682.84
6  2023         4119.43
7  2024         4626.73
8  2025         4711.30


In [10]:
# ============================================
# STEP 3 — CREATE SQL DATABASE
# ============================================

import sqlite3

# Create a database file in your current folder
conn = sqlite3.connect('oil_geopolitical.db')

# Load all 3 dataframes into the database as tables
brent.to_sql('brent_prices', conn, if_exists='replace', index=False)
iran_clean.to_sql('iran_production', conn, if_exists='replace', index=False)
events.to_sql('geopolitical_events', conn, if_exists='replace', index=False)

print("✅ Database created: oil_geopolitical.db")
print("\n📋 Tables in database:")

# Verify all tables were created
cursor = conn.cursor()
cursor.execute("SELECT name FROM sqlite_master WHERE type='table'")
tables = cursor.fetchall()
for table in tables:
    print(f"   → {table[0]}")

# Check row counts
print("\n📊 Row counts:")
for table in tables:
    cursor.execute(f"SELECT COUNT(*) FROM {table[0]}")
    count = cursor.fetchone()[0]
    print(f"   → {table[0]}: {count} rows")

✅ Database created: oil_geopolitical.db

📋 Tables in database:
   → brent_prices
   → iran_production
   → geopolitical_events

📊 Row counts:
   → brent_prices: 2345 rows
   → iran_production: 9 rows
   → geopolitical_events: 14 rows


In [12]:
# ============================================
# STEP 4 — SQL QUERIES FOR ANALYSIS
# ============================================

# Helper function to run SQL and see results nicely
def run_sql(query, description):
    print(f"\n{'='*50}")
    print(f"📌 {description}")
    print('='*50)
    result = pd.read_sql_query(query, conn)
    print(result.to_string(index=False))
    return result

# --- Query 1: Average oil price per year ---
q1 = run_sql("""
    SELECT 
        strftime('%Y', date) AS year,
        ROUND(AVG(price), 2) AS avg_price,
        ROUND(MIN(price), 2) AS min_price,
        ROUND(MAX(price), 2) AS max_price
    FROM brent_prices
    GROUP BY year
    ORDER BY year
""", "Average Brent Oil Price Per Year (2017-2026)")

# --- Query 2: Biggest price spikes (day over day) ---
q2 = run_sql("""
    SELECT 
        date,
        price,
        ROUND(price - LAG(price) OVER (ORDER BY date), 2) AS daily_change
    FROM brent_prices
    ORDER BY daily_change DESC
    LIMIT 10
""", "Top 10 Biggest Single-Day Price Increases")

# --- Query 3: Iran production vs price by year ---
q3 = run_sql("""
    SELECT 
        i.year,
        i.production_mbd AS iran_production,
        ROUND(AVG(b.price), 2) AS avg_brent_price
    FROM iran_production i
    LEFT JOIN brent_prices b 
        ON strftime('%Y', b.date) = CAST(i.year AS TEXT)
    GROUP BY i.year
    ORDER BY i.year
""", "Iran Production vs Average Brent Price by Year")

# --- Query 4: Price around geopolitical events ---
q4 = run_sql("""
    SELECT 
        event,
        severity,
        date
    FROM geopolitical_events
    ORDER BY date
""", "All Geopolitical Events Timeline")


📌 Average Brent Oil Price Per Year (2017-2026)
year  avg_price  min_price  max_price
2017      54.12      43.98      66.80
2018      71.33      50.57      86.07
2019      64.30      53.23      74.94
2020      41.96       9.12      70.25
2021      70.86      50.37      85.76
2022     100.93      76.02     133.18
2023      82.49      71.03      97.10
2024      80.52      70.31      93.12
2025      69.14      59.93      83.48
2026      79.98      61.08     121.88

📌 Top 10 Biggest Single-Day Price Increases
               date  price  daily_change
2026-03-12 00:00:00 102.38         11.40
2026-03-18 00:00:00 118.09          9.70
2022-03-17 00:00:00 113.50          8.89
2022-03-04 00:00:00 123.86          8.50
2026-03-27 00:00:00 121.47          8.08
2022-03-02 00:00:00 118.94          8.01
2022-03-21 00:00:00 122.29          7.97
2022-03-01 00:00:00 110.93          7.85
2026-03-20 00:00:00 118.42          7.37
2026-03-17 00:00:00 108.39          7.35

📌 Iran Production vs Average Brent Pr

In [13]:
# ============================================
# STEP 5 — SAVE CLEANED DATA FOR PHASE 2
# ============================================

# Save all query results as CSV files for Python analysis
q1.to_csv('yearly_avg_prices.csv', index=False)
q3.to_csv('iran_vs_price.csv', index=False)

# Save the clean master datasets too
brent.to_csv('brent_clean.csv', index=False)
iran_clean.to_csv('iran_clean.csv', index=False)
events.to_csv('events_clean.csv', index=False)

# Close database connection
conn.close()

print("✅ All files saved successfully!")
print("\n📁 Files created:")
print("   → yearly_avg_prices.csv")
print("   → iran_vs_price.csv")
print("   → brent_clean.csv")
print("   → iran_clean.csv")
print("   → events_clean.csv")
print("\n🎉 Phase 1 — SQL Complete!")
print("📊 Ready for Phase 2 — Python Analysis & Visualization")

✅ All files saved successfully!

📁 Files created:
   → yearly_avg_prices.csv
   → iran_vs_price.csv
   → brent_clean.csv
   → iran_clean.csv
   → events_clean.csv

🎉 Phase 1 — SQL Complete!
📊 Ready for Phase 2 — Python Analysis & Visualization
